In [1]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 6.33.0
Uninstalling protobuf-6.33.0:
  Successfully uninstalled protobuf-6.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.1 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [4]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import re
import seaborn as sns
import tokenize
import torch
import transformers

from datasets import Dataset

from math import ceil

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    logging
)

from tqdm.auto import tqdm

2026-01-13 04:25:25.164232: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768278325.351080      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768278325.408656      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


(OBS) Code block added to delete the SQL status of the checkpoint (it has 7 GBs, so it ocuppies much space on the output). Add this code block at the beginning of the script, right after the imports.

In [5]:
path = "/kaggle/working/state.db"

if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")


state.db not found


In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="macro")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [7]:
MODEL_NAME = "microsoft/unixcoder-base"
MAX_LENGTH = 192

In [8]:
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B"

training_path = "/kaggle/input/sample/training_sample_set.parquet"
validation_path = "/kaggle/input/sample/validation_sample_set.parquet"
test_sample_path = base_path + "/test_sample.parquet"
test_full_path = base_path + "/test.parquet"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_sample_df = pd.read_parquet(test_sample_path)
test_full_df = pd.read_parquet(test_full_path)

test_df = pd.merge(test_sample_df, test_full_df, on="code", how="inner")

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The functions below preprocess code samples and erase the comments.

In [10]:
def is_escaped(result, i):
    count = 0
    i -= 1
    while i >= 0 and result[i] == '\\':
        count += 1
        i -= 1
    return count % 2 == 1

In [11]:
def strip_jcg_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_verbatim_string = False

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if string_delimiter == '`':
                if c == '`':
                    string_delimiter = None
            elif string_delimiter == '"':
                if c == '"' and not is_escaped(code, i):
                    string_delimiter = None
            elif string_delimiter == "'":
                if c == "'" and not is_escaped(code, i):
                    string_delimiter = None
            i += 1
            continue

        if in_verbatim_string:
            if c == '"' and next_c == '"':
                i += 2
            else:
                if c == '"' and next_c != '"':
                    in_verbatim_string = False
                i += 1
            continue

        if c == '@' and next_c == '"':
            in_verbatim_string = True
            i += 2
            continue

        if c == "'" or c == '"' or c == '`':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [12]:
def strip_php_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_heredoc = False
    heredoc_id = None

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if i == 0 or code[i - 1] == '\n':
            line_start = i
        else:
            line_start = None

        if in_heredoc:
            if line_start is not None:
                j = line_start
                k = 0
                while j < n and k < len(heredoc_id) and code[j] == heredoc_id[k]:
                    j += 1
                    k += 1
                if k == len(heredoc_id) and (j == n or code[j] in (';', '\n')):
                    in_heredoc = False
                    heredoc_id = None
            i += 1
            continue

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c == '<' and code[i:i+3] == '<<<':
            j = i + 3

            while j < n and code[j].isspace():
                j += 1

            if j < n and code[j] in ("'", '"'):
                quote = code[j]
                j += 1
                start = j
                while j < n and code[j] != quote:
                    j += 1
                heredoc_id = code[start:j]
                j += 1
            else:
                start = j
                while j < n and (code[j].isalnum() or code[j] == '_'):
                    j += 1
                heredoc_id = code[start:j]

            in_heredoc = True
            i = j
            continue


        if c == "'" or c == '"':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '#':
            result[i] = ' '
            in_single_line_comment = True
            i += 1
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [13]:
def strip_python_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    string_delimiter = None
    in_comment = False

    while i < n:
        c = code[i]

        if in_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_comment = False
            i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c in ("'", '"'):
            string_delimiter = c
            i += 1
            continue

        if c == '#':
            result[i] = ' '
            in_comment = True
            i += 1
            continue

        i += 1

    return ''.join(result)

In [14]:
fallback_count = 0

def remove_python_docstrings(code: str) -> str:
    global fallback_count
    try:
        tokens = tokenize.generate_tokens(io.StringIO(code).readline)
        result = []
    
        scope_stack = [True]
    
        for tok in tokens:
            tok_type, tok_str, _, _, _ = tok
    
            if tok_type == tokenize.INDENT:
                scope_stack.append(True)
            elif tok_type == tokenize.DEDENT:
                scope_stack.pop()
            elif tok_type == tokenize.STRING and scope_stack[-1]:
                scope_stack[-1] = False
                continue
            elif tok_type not in (tokenize.NL, tokenize.NEWLINE):
                scope_stack[-1] = False
    
            result.append(tok)
    
        return tokenize.untokenize(result)

    except (IndentationError, SyntaxError, tokenize.TokenError):
        fallback_count += 1
        return code

In [15]:
def strip_python_comments_and_docstrings(code: str) -> str:
    code = strip_python_comments(code)
    code = remove_python_docstrings(code)
    return code

In [16]:
def normalize_whitespace(code: str) -> str:
    lines = code.splitlines()
    normalized = []

    for line in lines:
        stripped = line.rstrip()

        if stripped:
            m = re.match(r'^(\s*)(.*)$', stripped)
            indent, content = m.groups()
            content = re.sub(r' {2,}', ' ', content)
            normalized.append(indent + content)

    return '\n'.join(normalized)

In [17]:
def remove_jcg_comments(code: str) -> str:
    return normalize_whitespace(strip_jcg_comments(code))


def remove_php_comments(code: str) -> str:
    return normalize_whitespace(strip_php_comments(code))


def remove_python_comments(code: str) -> str:
    return normalize_whitespace(strip_python_comments_and_docstrings(code))

In [18]:
def clean_code(code: str, language: str) -> str:
    lang = language.lower() if isinstance(language, str) else ""
    if lang == "python":
        return remove_python_comments(code)
    elif lang == "php":
        return remove_php_comments(code)
    else:
        return remove_jcg_comments(code)

In [19]:
def preprocess_function(examples: pd.DataFrame):
    cleaned_code = [
        clean_code(code, lang)
        for code, lang in zip(examples["code"], examples["language"])
    ]

    examples["code"] = cleaned_code

    tokenized = tokenizer(
        examples["code"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    return tokenized

In [20]:
def binary_label(generator):
    return 0 if generator == "Human" else 1

for df in (training_df, validation_df, test_df):
    for col in ["label", "labels", "binary_label"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    df["binary_label"] = df["generator"].apply(binary_label).astype("int64")

(OBS) It is good practice to set the format to torch after you tokenize the datasets. Add the .set_format("torch") instructions right after you tokenized the datasets.

In [21]:
columns_to_remove = ["binary_label", "code", "generator", "language", "__index_level_0__"]

training_dataset = Dataset.from_pandas(training_df, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

training_dataset = training_dataset.add_column(
    "labels",
    [int(x) for x in training_df["binary_label"].tolist()]
)
validation_dataset = validation_dataset.add_column(
    "labels",
    [int(x) for x in validation_df["binary_label"].tolist()]
)
test_dataset = test_dataset.add_column(
    "labels",
    [int(x) for x in test_df["binary_label"].tolist()]
)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)
test_tokenized_set = test_dataset.map(preprocess_function, batched=True)

training_tokenized_set = training_tokenized_set.remove_columns(
    [c for c in columns_to_remove if c in training_tokenized_set.column_names]
)
validation_tokenized_set = validation_tokenized_set.remove_columns(
    [c for c in columns_to_remove if c in validation_tokenized_set.column_names]
)
test_tokenized_set = test_tokenized_set.remove_columns(
    [c for c in columns_to_remove if c in test_tokenized_set.column_names]
)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")
test_tokenized_set.set_format("torch")

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nRunning on device:", device)
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))


Running on device: cuda
1
Tesla T4


In [23]:
print("Docstring fallback count:", fallback_count)

Docstring fallback count: 872


In [24]:
logging.set_verbosity_info()

In [25]:
class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            tqdm.write(f"Step {state.global_step}/{state.max_steps}")

In [26]:
binary_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

binary_model.to(device)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--microsoft--unixcoder-base/snapshots/5604afdc964f6c53782a6813140ade5216b99006/config.json
Model config RobertaConfig {
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 1026,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.53.3",
  "type_vocab_size": 10,
  "use_cache": true,
  "vocab_size": 51416
}



pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--microsoft--unixcoder-base/snapshots/5604afdc964f6c53782a6813140ade5216b99006/pytorch_model.bin
Attempting to create safetensors variant
Some weights of the model checkpoint at microsoft/unixcoder-base were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the mode

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(51416, 768, padding_idx=1)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
      (token_type_embeddings): Embedding(10, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
       

In [27]:
binary_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_a",
    seed=42,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

PyTorch: setting up devices


In [28]:
binary_trainer = Trainer(
    model=binary_model,
    args=binary_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
                   early_stopping_patience=1,
                   early_stopping_threshold=0.001
               ),
               TrainingProgressCallback()]
)

Using auto half precision backend


In [29]:
binary_trainer.train()

***** Running training *****
  Num examples = 300,000
  Num Epochs = 3
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 2
  Total optimization steps = 14,064
  Number of trainable parameters = 125,931,266


Step 1/14064


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision,Recall
1,0.100600,0.101214,0.913518,0.965750,0.965004,0.965750
2,0.069100,0.088003,0.923878,0.968950,0.968988,0.968950
3,0.040600,0.114403,0.919964,0.966850,0.967472,0.966850


Step 2/14064
Step 3/14064
Step 4/14064
Step 5/14064
Step 6/14064
Step 7/14064
Step 8/14064
Step 9/14064
Step 10/14064
Step 11/14064
Step 12/14064
Step 13/14064
Step 14/14064
Step 15/14064
Step 16/14064
Step 17/14064
Step 18/14064
Step 19/14064
Step 20/14064
Step 21/14064
Step 22/14064
Step 23/14064
Step 24/14064
Step 25/14064
Step 26/14064
Step 27/14064
Step 28/14064
Step 29/14064
Step 30/14064
Step 31/14064
Step 32/14064
Step 33/14064
Step 34/14064
Step 35/14064
Step 36/14064
Step 37/14064
Step 38/14064
Step 39/14064
Step 40/14064
Step 41/14064
Step 42/14064
Step 43/14064
Step 44/14064
Step 45/14064
Step 46/14064
Step 47/14064
Step 48/14064
Step 49/14064
Step 50/14064
Step 51/14064
Step 52/14064
Step 53/14064
Step 54/14064
Step 55/14064
Step 56/14064
Step 57/14064
Step 58/14064
Step 59/14064
Step 60/14064
Step 61/14064
Step 62/14064
Step 63/14064
Step 64/14064
Step 65/14064
Step 66/14064
Step 67/14064
Step 68/14064
Step 69/14064
Step 70/14064
Step 71/14064
Step 72/14064
Step 73/14064



***** Running Evaluation *****
  Num examples = 60000
  Batch size = 64
Saving model checkpoint to /kaggle/working/checkpoints_a/checkpoint-4688
Configuration saved in /kaggle/working/checkpoints_a/checkpoint-4688/config.json
Model weights saved in /kaggle/working/checkpoints_a/checkpoint-4688/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_a/checkpoint-4688/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_a/checkpoint-4688/special_tokens_map.json


Step 4689/14064
Step 4690/14064
Step 4691/14064
Step 4692/14064
Step 4693/14064
Step 4694/14064
Step 4695/14064
Step 4696/14064
Step 4697/14064
Step 4698/14064
Step 4699/14064
Step 4700/14064
Step 4701/14064
Step 4702/14064
Step 4703/14064
Step 4704/14064
Step 4705/14064
Step 4706/14064
Step 4707/14064
Step 4708/14064
Step 4709/14064
Step 4710/14064
Step 4711/14064
Step 4712/14064
Step 4713/14064
Step 4714/14064
Step 4715/14064
Step 4716/14064
Step 4717/14064
Step 4718/14064
Step 4719/14064
Step 4720/14064
Step 4721/14064
Step 4722/14064
Step 4723/14064
Step 4724/14064
Step 4725/14064
Step 4726/14064
Step 4727/14064
Step 4728/14064
Step 4729/14064
Step 4730/14064
Step 4731/14064
Step 4732/14064
Step 4733/14064
Step 4734/14064
Step 4735/14064
Step 4736/14064
Step 4737/14064
Step 4738/14064
Step 4739/14064
Step 4740/14064
Step 4741/14064
Step 4742/14064
Step 4743/14064
Step 4744/14064
Step 4745/14064
Step 4746/14064
Step 4747/14064
Step 4748/14064
Step 4749/14064
Step 4750/14064
Step 475


***** Running Evaluation *****
  Num examples = 60000
  Batch size = 64
Saving model checkpoint to /kaggle/working/checkpoints_a/checkpoint-9376
Configuration saved in /kaggle/working/checkpoints_a/checkpoint-9376/config.json
Model weights saved in /kaggle/working/checkpoints_a/checkpoint-9376/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_a/checkpoint-9376/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_a/checkpoint-9376/special_tokens_map.json


Step 9377/14064
Step 9378/14064
Step 9379/14064
Step 9380/14064
Step 9381/14064
Step 9382/14064
Step 9383/14064
Step 9384/14064
Step 9385/14064
Step 9386/14064
Step 9387/14064
Step 9388/14064
Step 9389/14064
Step 9390/14064
Step 9391/14064
Step 9392/14064
Step 9393/14064
Step 9394/14064
Step 9395/14064
Step 9396/14064
Step 9397/14064
Step 9398/14064
Step 9399/14064
Step 9400/14064
Step 9401/14064
Step 9402/14064
Step 9403/14064
Step 9404/14064
Step 9405/14064
Step 9406/14064
Step 9407/14064
Step 9408/14064
Step 9409/14064
Step 9410/14064
Step 9411/14064
Step 9412/14064
Step 9413/14064
Step 9414/14064
Step 9415/14064
Step 9416/14064
Step 9417/14064
Step 9418/14064
Step 9419/14064
Step 9420/14064
Step 9421/14064
Step 9422/14064
Step 9423/14064
Step 9424/14064
Step 9425/14064
Step 9426/14064
Step 9427/14064
Step 9428/14064
Step 9429/14064
Step 9430/14064
Step 9431/14064
Step 9432/14064
Step 9433/14064
Step 9434/14064
Step 9435/14064
Step 9436/14064
Step 9437/14064
Step 9438/14064
Step 943


***** Running Evaluation *****
  Num examples = 60000
  Batch size = 64
Saving model checkpoint to /kaggle/working/checkpoints_a/checkpoint-14064
Configuration saved in /kaggle/working/checkpoints_a/checkpoint-14064/config.json
Model weights saved in /kaggle/working/checkpoints_a/checkpoint-14064/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_a/checkpoint-14064/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_a/checkpoint-14064/special_tokens_map.json
Deleting older checkpoint [/kaggle/working/checkpoints_a/checkpoint-4688] due to args.save_total_limit


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from /kaggle/working/checkpoints_a/checkpoint-9376 (score: 0.9238775430633805).


TrainOutput(global_step=14064, training_loss=0.08522341904718857, metrics={'train_runtime': 8866.0854, 'train_samples_per_second': 101.51, 'train_steps_per_second': 1.586, 'total_flos': 8.8799981184e+16, 'train_loss': 0.08522341904718857, 'epoch': 3.0})

In [30]:
with open("taska_training_results.pkl", "wb") as f:
    pickle.dump(binary_trainer.state.log_history, f)

In [31]:
llm_training_df = training_df[training_df.generator != "Human"].copy()
llm_validation_df = validation_df[validation_df.generator != "Human"].copy()
llm_test_df = test_df[test_df.generator != "Human"].copy()

In [32]:
for df in (llm_training_df, llm_validation_df, llm_test_df):
    for col in ["label", "labels", "binary_label"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

In [38]:
LLM_FAMILIES = [
    "deepseek",
    "qwen",
    "01-ai",
    "bigcode",
    "gemma",
    "phi",
    "meta-llama",
    "ibm-granite",
    "mistral",
    "openai",
]

In [39]:
FAMILY_KEYWORDS = {
    "deepseek": ["deepseek"],
    "qwen": ["qwen"],
    "01-ai": ["01-ai", "yi-coder", "yi coder"],
    "bigcode": ["bigcode", "starcoder"],
    "gemma": ["gemma", "codegemma"],
    "phi": ["phi"],
    "meta-llama": ["llama", "meta-llama"],
    "ibm-granite": ["granite"],
    "mistral": ["mistral", "mixtral", "devstral"],
    "openai": ["gpt", "openai"],
}

In [40]:
def map_to_family(generator_name: str) -> str:
    if not isinstance(generator_name, str):
        return None
    
    name = generator_name.lower()
    
    for family, keywords in FAMILY_KEYWORDS.items():
        for keyword in keywords:
            if keyword in name:
                return family
    
    raise ValueError(f"Unmapped generator: {generator_name}")

In [43]:
for df in (llm_training_df, llm_validation_df, llm_test_df):
    df["llm_family"] = df["generator"].apply(map_to_family)

In [44]:
label2id = {label: i for i, label in enumerate(LLM_FAMILIES)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)

In [47]:
for df in (llm_training_df, llm_validation_df, llm_test_df):
    df["labels"] = df["llm_family"].map(label2id).astype("int64")

In [50]:
columns_to_remove = ["generator", "language", "code", "llm_family", "__index_level_0__"]

llm_training_dataset = Dataset.from_pandas(llm_training_df, preserve_index=False)
llm_validation_dataset = Dataset.from_pandas(llm_validation_df, preserve_index=False)
llm_test_dataset = Dataset.from_pandas(llm_test_df, preserve_index=False)

llm_training_tokenized_set = llm_training_dataset.map(preprocess_function, batched=True)
llm_validation_tokenized_set = llm_validation_dataset.map(preprocess_function, batched=True)
llm_test_tokenized_set = llm_test_dataset.map(preprocess_function, batched=True)

for ds in (llm_training_tokenized_set, llm_validation_tokenized_set, llm_test_tokenized_set):
    ds = ds.remove_columns([c for c in columns_to_remove if c in ds.column_names])


llm_training_tokenized_set.set_format("torch")
llm_validation_tokenized_set.set_format("torch")
llm_test_tokenized_set.set_format("torch")

Map:   0%|          | 0/34742 [00:00<?, ? examples/s]

Map:   0%|          | 0/6906 [00:00<?, ? examples/s]

Map:   0%|          | 0/526 [00:00<?, ? examples/s]

In [51]:
llm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

llm_model.to(device)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--microsoft--unixcoder-base/snapshots/5604afdc964f6c53782a6813140ade5216b99006/config.json
Model config RobertaConfig {
  "architectures": [
    "RobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "deepseek",
    "1": "qwen",
    "2": "01-ai",
    "3": "bigcode",
    "4": "gemma",
    "5": "phi",
    "6": "meta-llama",
    "7": "ibm-granite",
    "8": "mistral",
    "9": "openai"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "01-ai": 2,
    "bigcode": 3,
    "deepseek": 0,
    "gemma": 4,
    "ibm-granite": 7,
    "meta-llama": 6,
    "mistral": 8,
    "openai": 9,
    "phi": 5,
    "qwen": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings":

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(51416, 768, padding_idx=1)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
      (token_type_embeddings): Embedding(10, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
       

Safetensors PR exists


In [53]:
counts = torch.tensor(
    llm_training_df["labels"].value_counts().sort_index().values,
    dtype=torch.float
)

class_weights = (1.0 / counts).sqrt()
class_weights = class_weights / class_weights.sum()

In [54]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_function = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_function(logits, labels)

        return (loss, outputs) if return_outputs else loss

(OBS) In the TrainingArguments parameters, you have to set the following:
- output_dir="/kaggle/working/checkpoints" (this is the Kaggle notebook folder that stores the output files; you will need to have persistent data, to achieve that the checkpoints must be saved in this folder);
- num_train_epochs=3;
- load_best_model_at_end=True (for best results);
- eval_strategy="epoch";
- save_strategy="epoch";
- save_total_limit=2 (or 3, you set here the last number of checkpoints that remain saved);
- logging_dir="/kaggle/working/logs" (optional, stores log information);
- logging_steps=3000 (optional, add only if you put logging_dir too);

In [55]:
llm_training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_b",
    seed=42,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir="/kaggle/working/logs",
    logging_steps=3000,
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

PyTorch: setting up devices


In [56]:
llm_trainer = WeightedTrainer(
    model=llm_model,
    args=llm_training_args,
    train_dataset=llm_training_tokenized_set,
    eval_dataset=llm_validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
                   early_stopping_patience=1,
                   early_stopping_threshold=0.001
               ),
               TrainingProgressCallback()]
)

Using auto half precision backend


(OBS) The first execution will have just trainer.train(), because initially you won't have any checkpoints, you start from 0 with the training. After that, you need to replace this instruction with trainer.train(resume_from_checkpoint=True), such that it resumes the training from the last checkpoint.

In [57]:
llm_trainer.train()

The following columns in the Training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: llm_family, language, code, generator. If llm_family, language, code, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 34,742
  Num Epochs = 5
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 4
  Total optimization steps = 2,715
  Number of trainable parameters = 125,937,418


Step 1/2715


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision,Recall
1,No log,1.771641,0.299193,0.374023,0.351651,0.374023
2,No log,1.707178,0.348965,0.396467,0.396369,0.396467
3,No log,1.760222,0.366662,0.401825,0.409051,0.401825
4,No log,1.964666,0.379791,0.409933,0.404236,0.409933
5,No log,2.045655,0.374955,0.400376,0.398571,0.400376


Step 2/2715
Step 3/2715
Step 4/2715
Step 5/2715
Step 6/2715
Step 7/2715
Step 8/2715
Step 9/2715
Step 10/2715
Step 11/2715
Step 12/2715
Step 13/2715
Step 14/2715
Step 15/2715
Step 16/2715
Step 17/2715
Step 18/2715
Step 19/2715
Step 20/2715
Step 21/2715
Step 22/2715
Step 23/2715
Step 24/2715
Step 25/2715
Step 26/2715
Step 27/2715
Step 28/2715
Step 29/2715
Step 30/2715
Step 31/2715
Step 32/2715
Step 33/2715
Step 34/2715
Step 35/2715
Step 36/2715
Step 37/2715
Step 38/2715
Step 39/2715
Step 40/2715
Step 41/2715
Step 42/2715
Step 43/2715
Step 44/2715
Step 45/2715
Step 46/2715
Step 47/2715
Step 48/2715
Step 49/2715
Step 50/2715
Step 51/2715
Step 52/2715
Step 53/2715
Step 54/2715
Step 55/2715
Step 56/2715
Step 57/2715
Step 58/2715
Step 59/2715
Step 60/2715
Step 61/2715
Step 62/2715
Step 63/2715
Step 64/2715
Step 65/2715
Step 66/2715
Step 67/2715
Step 68/2715
Step 69/2715
Step 70/2715
Step 71/2715
Step 72/2715
Step 73/2715
Step 74/2715
Step 75/2715
Step 76/2715
Step 77/2715
Step 78/2715
Step 79

The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: llm_family, language, code, generator. If llm_family, language, code, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 6906
  Batch size = 32
Saving model checkpoint to /kaggle/working/checkpoints_b/checkpoint-543
Configuration saved in /kaggle/working/checkpoints_b/checkpoint-543/config.json
Model weights saved in /kaggle/working/checkpoints_b/checkpoint-543/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_b/checkpoint-543/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_b/checkpoint-543/special_tokens_map.json


...

Step 1630/2715
Step 1631/2715
Step 1632/2715
Step 1633/2715
Step 1634/2715
Step 1635/2715
Step 1636/2715
Step 1637/2715
Step 1638/2715
Step 1639/2715
Step 1640/2715
Step 1641/2715
Step 1642/2715
Step 1643/2715
Step 1644/2715
Step 1645/2715
Step 1646/2715
Step 1647/2715
Step 1648/2715
Step 1649/2715
Step 1650/2715
Step 1651/2715
Step 1652/2715
Step 1653/2715
Step 1654/2715
Step 1655/2715
Step 1656/2715
Step 1657/2715
Step 1658/2715
Step 1659/2715
Step 1660/2715
Step 1661/2715
Step 1662/2715
Step 1663/2715
Step 1664/2715
Step 1665/2715
Step 1666/2715
Step 1667/2715
Step 1668/2715
Step 1669/2715
Step 1670/2715
Step 1671/2715
Step 1672/2715
Step 1673/2715
Step 1674/2715
Step 1675/2715
Step 1676/2715
Step 1677/2715
Step 1678/2715
Step 1679/2715
Step 1680/2715
Step 1681/2715
Step 1682/2715
Step 1683/2715
Step 1684/2715
Step 1685/2715
Step 1686/2715
Step 1687/2715
Step 1688/2715
Step 1689/2715
Step 1690/2715
Step 1691/2715
Step 1692/2715
Step 1693/2715
Step 1694/2715
Step 1695/2715
Step 1696/

The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: llm_family, language, code, generator. If llm_family, language, code, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 6906
  Batch size = 32
Saving model checkpoint to /kaggle/working/checkpoints_b/checkpoint-2172
Configuration saved in /kaggle/working/checkpoints_b/checkpoint-2172/config.json
Model weights saved in /kaggle/working/checkpoints_b/checkpoint-2172/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_b/checkpoint-2172/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_b/checkpoint-2172/special_tokens_map.json
Deleting older checkpoint [/kaggle/working/checkpoints_b/checkpoint-1086] due to args.save_total_limit


Step 2173/2715
Step 2174/2715
Step 2175/2715
Step 2176/2715
Step 2177/2715
Step 2178/2715
Step 2179/2715
Step 2180/2715
Step 2181/2715
Step 2182/2715
Step 2183/2715
Step 2184/2715
Step 2185/2715
Step 2186/2715
Step 2187/2715
Step 2188/2715
Step 2189/2715
Step 2190/2715
Step 2191/2715
Step 2192/2715
Step 2193/2715
Step 2194/2715
Step 2195/2715
Step 2196/2715
Step 2197/2715
Step 2198/2715
Step 2199/2715
Step 2200/2715
Step 2201/2715
Step 2202/2715
Step 2203/2715
Step 2204/2715
Step 2205/2715
Step 2206/2715
Step 2207/2715
Step 2208/2715
Step 2209/2715
Step 2210/2715
Step 2211/2715
Step 2212/2715
Step 2213/2715
Step 2214/2715
Step 2215/2715
Step 2216/2715
Step 2217/2715
Step 2218/2715
Step 2219/2715
Step 2220/2715
Step 2221/2715
Step 2222/2715
Step 2223/2715
Step 2224/2715
Step 2225/2715
Step 2226/2715
Step 2227/2715
Step 2228/2715
Step 2229/2715
Step 2230/2715
Step 2231/2715
Step 2232/2715
Step 2233/2715
Step 2234/2715
Step 2235/2715
Step 2236/2715
Step 2237/2715
Step 2238/2715
Step 2239/

The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: llm_family, language, code, generator. If llm_family, language, code, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 6906
  Batch size = 32
Saving model checkpoint to /kaggle/working/checkpoints_b/checkpoint-2715
Configuration saved in /kaggle/working/checkpoints_b/checkpoint-2715/config.json
Model weights saved in /kaggle/working/checkpoints_b/checkpoint-2715/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_b/checkpoint-2715/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_b/checkpoint-2715/special_tokens_map.json
Deleting older checkpoint [/kaggle/working/checkpoints_b/checkpoint-1629] due to args.save_total_limit


Training completed. Do not forget to share your model 

TrainOutput(global_step=2715, training_loss=1.3447720634783609, metrics={'train_runtime': 1963.275, 'train_samples_per_second': 88.48, 'train_steps_per_second': 1.383, 'total_flos': 1.714061413580544e+16, 'train_loss': 1.3447720634783609, 'epoch': 5.0})

In [58]:
with open("taskb_training_results.pkl", "wb") as f:
    pickle.dump(llm_trainer.state.log_history, f)

In [82]:
binary_trainer.save_model("/kaggle/working/final_binary_model")
llm_trainer.save_model("/kaggle/working/final_llm_model")
tokenizer.save_pretrained("/kaggle/working/final_tokenizer")

Saving model checkpoint to /kaggle/working/final_binary_model
Configuration saved in /kaggle/working/final_binary_model/config.json
Model weights saved in /kaggle/working/final_binary_model/model.safetensors
tokenizer config file saved in /kaggle/working/final_binary_model/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_binary_model/special_tokens_map.json
Saving model checkpoint to /kaggle/working/final_llm_model
Configuration saved in /kaggle/working/final_llm_model/config.json
Model weights saved in /kaggle/working/final_llm_model/model.safetensors
tokenizer config file saved in /kaggle/working/final_llm_model/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_llm_model/special_tokens_map.json
tokenizer config file saved in /kaggle/working/final_tokenizer/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_tokenizer/special_tokens_map.json


('/kaggle/working/final_tokenizer/tokenizer_config.json',
 '/kaggle/working/final_tokenizer/special_tokens_map.json',
 '/kaggle/working/final_tokenizer/vocab.json',
 '/kaggle/working/final_tokenizer/merges.txt',
 '/kaggle/working/final_tokenizer/added_tokens.json',
 '/kaggle/working/final_tokenizer/tokenizer.json')

In [75]:
def tokenize_test_set(examples: pd.DataFrame):
    tokenizer(
        text_target=examples["code"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [80]:
test_full_dataset = Dataset.from_pandas(test_full_df, preserve_index=False)

test_full_tokenized_set = test_full_dataset.map(tokenize_test_set, batched=True)

test_full_tokenized_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

ValueError: Columns ['input_ids', 'attention_mask'] not in the dataset. Current columns in the dataset: ['ID', 'code']

In [81]:
binary_evaluation = binary_trainer.evaluate(test_full_tokenized_set)
print("Evaluation for task A: ", binary_evaluation)

binary_preds = np.argmax(
    binary_trainer.predict(test_full_tokenized_set).predictions, axis=1
)

llm_evaluation = llm_trainer.evaluate(test_full_tokenized_set)
print("Evaluation for task B: ", llm_evaluation)

llm_preds = np.argmax(
    llm_trainer.predict(test_full_tokenized_set).predictions, axis=1
)

final_preds = []
llm_index = 0
for p in binary_preds:
    if p == 0:
        final_preds.append("Human")
    else:
        final_preds.append(id2label[llm_preds[llm_index]])
        llm_index += 1


The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: ID, code. If ID, code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.


ValueError: No columns in the dataset match the model's forward method signature: (input_ids, attention_mask, token_type_ids, position_ids, head_mask, inputs_embeds, labels, output_attentions, output_hidden_states, return_dict, labels, label, label_ids). The following columns have been ignored: [ID, code]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [63]:
pd.DataFrame({
    "ID": test_df["ID"],
    "label": binary_preds
}).to_csv("taskA_predictions.csv", index=False)

pd.DataFrame({
    "ID": test_df["ID"],
    "label": final_preds
}).to_csv("taskB_predictions.csv", index=False)